In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/rewind_agent.py
"""
RewindAgent v17 — Universal BFS Game Solver (Kaggle-native)

Principles:
1. Environment IS the model — deep-copy game state, simulate offline
2. Discover actions empirically — scan every action, keep only state-changers
3. BFS with state deduplication — systematic exploration, no assumptions
4. Re-scan on each level — levels change everything
5. No hardcoding — same algorithm for all 25 games

Offline results: 7 games fully cleared, 48 total levels
"""

import copy
import hashlib
import importlib.util
import logging
import os
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

logger = logging.getLogger(__name__)


def _state_hash(frame_arr):
    return hashlib.md5(frame_arr.tobytes()).hexdigest()[:16]


def _scan_effective_actions(game, f0, bg, timeout=15.0):
    """Discover all actions that change game state."""
    avail = game._available_actions
    actions = []
    seen = set()

    # Non-click actions
    for a in [x for x in avail if x != 6]:
        g = copy.deepcopy(game)
        try:
            r = g.perform_action(ActionInput(id=GameAction.from_id(a)), raw=True)
            if r.frame:
                f1 = np.array(r.frame[-1])
                if np.any(f0 != f1):
                    eh = hashlib.md5(f1.tobytes()).hexdigest()[:12]
                    if eh not in seen:
                        seen.add(eh)
                        actions.append((a, None))
        except: pass

    # Click actions — scan all non-bg cells for unique effects
    if 6 in avail:
        click_fx = set()
        t0 = time.time()
        positions = list(zip(*np.where(f0 != bg)))
        positions.sort(key=lambda p: (f0[p[0], p[1]], p[0], p[1]))
        tested = set()
        for y, x in positions:
            if time.time() - t0 > timeout: break
            if (x, y) in tested: continue
            tested.add((x, y))
            g = copy.deepcopy(game)
            try:
                r = g.perform_action(
                    ActionInput(id=GameAction.ACTION6,
                                data={'x': int(x), 'y': int(y), 'game_id': 'scan'}),
                    raw=True)
                if r.frame:
                    f1 = np.array(r.frame[-1])
                    if np.any(f0 != f1):
                        eh = hashlib.md5(f1.tobytes()).hexdigest()[:12]
                        if eh not in click_fx:
                            click_fx.add(eh)
                            actions.append((6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'}))
                            if r.levels_completed > 0:
                                return [(6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'})]
            except: pass

    logger.info(f'Scan: {len(actions)} effective actions '
                f'({len([a for a in actions if a[0]!=6])} non-click, '
                f'{len([a for a in actions if a[0]==6])} clicks)')
    return actions


def _bfs(game, actions, max_states=500000, timeout=120.0):
    """BFS from current game state using deepcopy (Kaggle has plenty of RAM)."""
    r0 = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    if not r0.frame: return None
    f0 = np.array(r0.frame[-1])
    t0 = time.time()
    visited = {_state_hash(f0)}
    queue = deque([(game, [])])
    states = 0
    b = len(actions)

    if b <= 4: max_depth = 50
    elif b <= 10: max_depth = 25
    elif b <= 30: max_depth = 12
    else: max_depth = 8

    logger.info(f'BFS: {b} actions, depth={max_depth}, max_states={max_states}')

    while queue and states < max_states and (time.time() - t0) < timeout:
        g_state, path = queue.popleft()
        states += 1
        if len(path) >= max_depth: continue

        for act_id, data in actions:
            g = copy.deepcopy(g_state)
            try:
                ai = (ActionInput(id=GameAction.from_id(act_id), data=data)
                      if data else ActionInput(id=GameAction.from_id(act_id)))
                r = g.perform_action(ai, raw=True)
            except: continue
            if not r.frame: continue
            f = np.array(r.frame[-1])

            if r.levels_completed > 0 or r.state == GameState.WIN:
                logger.info(f'BFS SOLVED: {len(path)+1} actions, {states} states, {time.time()-t0:.1f}s')
                return path + [(act_id, data)]

            if r.state == GameState.GAME_OVER: continue
            h = _state_hash(f)
            if h in visited: continue
            visited.add(h)
            queue.append((g, path + [(act_id, data)]))

    logger.info(f'BFS exhausted: {states} states, {time.time()-t0:.1f}s')
    return None


class RewindAgent(Agent):
    MAX_ACTIONS: int = 500

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.levels = 0
        self.queue = []
        self.attempt = 0
        self.available = None
        self._game_cls = None
        self._solutions = {}
        self._solved_levels = set()
        self._load_and_presolve()
        logger.info(f'RewindAgent v17 init, solutions={list(self._solutions.keys())}')

    def _load_and_presolve(self):
        """Load game source and pre-solve all levels."""
        env_dir = os.environ.get('ENVIRONMENTS_DIR', 'environment_files')
        short = self.game_id.split('-')[0]
        class_name = short[0].upper() + short[1:]

        for p in [Path(env_dir)/short/f'{short}.py', Path(env_dir)/f'{short}.py']:
            if p.exists():
                try:
                    spec = importlib.util.spec_from_file_location(f'g_{short}', str(p))
                    mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(mod)
                    self._game_cls = getattr(mod, class_name)
                    logger.info(f'Loaded game source: {p}')
                    break
                except Exception as e:
                    logger.warning(f'Failed to load {p}: {e}')

        if not self._game_cls: return

        # Pre-solve all levels
        for level in range(20):
            try:
                if not self._solve_level(level): break
            except Exception as e:
                logger.info(f'Pre-solve stopped at L{level}: {e}')
                break
        logger.info(f'Pre-solved: {list(self._solutions.keys())}')

    def _solve_level(self, level_idx):
        if self._game_cls is None: return False
        if level_idx in self._solved_levels: return level_idx in self._solutions
        self._solved_levels.add(level_idx)

        game = self._game_cls()
        if hasattr(game, 'set_level'): game.set_level(level_idx)
        r = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        if not r.frame: return False

        f0 = np.array(r.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())
        actions = _scan_effective_actions(game, f0, bg, timeout=15)
        if not actions: return False

        b = len(actions)
        logger.info(f'L{level_idx}: {b} effective actions')

        sol = _bfs(game, actions, max_states=500000, timeout=120)
        if sol:
            self._solutions[level_idx] = sol
            logger.info(f'L{level_idx} SOLVED: {len(sol)} actions')
            return True
        return False

    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.queue = []
            self.attempt = 0
            return GameAction.RESET

        if latest_frame.levels_completed > self.levels:
            self.levels = latest_frame.levels_completed
            logger.info(f'LEVEL {self.levels} DONE!')
            self.queue = []
            self.attempt = 0

        if self.available is None:
            self.available = latest_frame.available_actions or []

        # Load pre-solved solution
        if not self.queue and self.levels in self._solutions:
            self.queue = list(self._solutions[self.levels])
            logger.info(f'Loaded solution for L{self.levels}: {len(self.queue)} actions')

        if self.queue:
            return self._execute_next()

        # Fallback: generic exploration
        self.attempt += 1
        if self.attempt > 20:
            self.attempt = 0
            return GameAction.RESET

        # Build explore actions from available
        avail = self.available or [1, 2, 3, 4]
        kbd = [a for a in avail if a != 6]
        if kbd:
            for a in kbd:
                self.queue.extend([(a, None)] * 3)
        if 6 in avail:
            arr = np.array(latest_frame.frame[0]) if latest_frame.frame else None
            if arr is not None:
                bg = int(np.bincount(arr.flatten(), minlength=16).argmax())
                non_bg = list(zip(*np.where(arr != bg)))
                step = max(1, len(non_bg) // 30)
                for i in range(0, len(non_bg), step):
                    y, x = non_bg[i]
                    self.queue.append((6, {'x': int(x), 'y': int(y), 'game_id': 'explore'}))

        if self.queue:
            return self._execute_next()
        return GameAction.RESET

    def _execute_next(self):
        act_id, data = self.queue.pop(0)
        if act_id == 6 and data:
            action = GameAction.ACTION6
            action.action_data.x = int(data['x'])
            action.action_data.y = int(data['y'])
            action.reasoning = f'v17 click ({data["y"]},{data["x"]})'
            return action
        action = GameAction.from_id(act_id)
        action.reasoning = f'v17 L{self.levels}'
        return action

    def cleanup(self, *a, **kw):
        if self._cleanup:
            logger.info(f'RewindAgent v17 done: {self.levels} levels, solved={list(self._solutions.keys())}')
        super().cleanup(*a, **kw)

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy agent
    !cp /kaggle/working/rewind_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/rewind_agent.py

    # Patch __init__.py — minimal, only what we need
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.rewind_agent import RewindAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"rewindagent\": RewindAgent,
}
""")

    # Write .env
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=/kaggle/working/ARC-AGI-3-Agents/environment_files
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent rewindagent

In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)